In [ ]:
import time
import requests
import pandas as pd

api_key = "Апи ключ. Конфиденциальная информация"

input_file = "top1000.csv"
output_file = "games_youtube_wide.csv"

search_url = "https://www.googleapis.com/youtube/v3/search"
videos_url = "https://www.googleapis.com/youtube/v3/videos"

top_n = 5
max_results = 25
sleep_time = 0.15

In [ ]:
def get_video_ids(game_name):
    params = {
        "part": "snippet",
        "q": game_name,
        "type": "video",
        "maxResults": max_results,
        "key": api_key
    }

    response = requests.get(search_url, params=params)
    response.raise_for_status()
    data = response.json()

    ids = []
    for item in data.get("items", []):
        if "videoId" in item.get("id", {}):
            ids.append(item["id"]["videoId"])

    return ids

In [ ]:
def get_videos_info(video_ids):
    if not video_ids:
        return []

    params = {
        "part": "snippet,statistics,contentDetails",
        "id": ",".join(video_ids),
        "key": api_key
    }

    response = requests.get(videos_url, params=params)
    response.raise_for_status()
    data = response.json()

    videos = []

    for item in data.get("items", []):
        snippet = item.get("snippet", {})
        stats = item.get("statistics", {})
        details = item.get("contentDetails", {})

        video_info = {
            "id": item.get("id"),
            "title": snippet.get("title"),
            "channel": snippet.get("channelTitle"),
            "published_at": snippet.get("publishedAt"),
            "description": snippet.get("description"),
            "duration": details.get("duration"),
            "views": stats.get("viewCount"),
            "likes": stats.get("likeCount"),
            "comments": stats.get("commentCount")
        }

        videos.append(video_info)

    return videos

In [ ]:
try:
    games = pd.read_csv(input_file)
    final_rows = []

    for _, game in games.iterrows():
        game_name = str(game["name"])

        video_ids = get_video_ids(game_name)
        time.sleep(sleep_time)

        videos = get_videos_info(video_ids)
        time.sleep(sleep_time)

        videos = sorted(
            videos,
            key=lambda x: int(x["views"]) if x["views"] else 0,
            reverse=True
        )

        top_videos = videos[:top_n]
        row = game.to_dict()

        for i in range(top_n):
            if i < len(top_videos):
                video = top_videos[i]
            else:
                video = {
                    "id": None,
                    "title": None,
                    "channel": None,
                    "published_at": None,
                    "description": None,
                    "duration": None,
                    "views": None,
                    "likes": None,
                    "comments": None
                }

            num = i + 1

            row[f"yt_video_{num}_id"] = video["id"]
            row[f"yt_video_{num}_title"] = video["title"]
            row[f"yt_video_{num}_channel"] = video["channel"]
            row[f"yt_video_{num}_published_at"] = video["published_at"]
            row[f"yt_video_{num}_description"] = video["description"]
            row[f"yt_video_{num}_duration"] = video["duration"]
            row[f"yt_video_{num}_views"] = video["views"]
            row[f"yt_video_{num}_likes"] = video["likes"]
            row[f"yt_video_{num}_comments"] = video["comments"]

        final_rows.append(row)

    result = pd.DataFrame(final_rows)
    result.to_csv(output_file, index=False)

    print("Код успешно выполнился. Файл games_youtube_wide.csv создан")

except Exception as error:
    print("Код не выполнился")
    print("Ошибка:", error)